# Proyecto: Construcción del Proceso ETL — Aseguradora RaSA
**Curso:** Modelado de Datos y ETL (MIID-4103) — Maestría en Inteligencia Analítica de Datos, Universidad de los Andes  
**Equipo:** Andrés Argumero, Mateo Losada, José Torres  

---

## 1. Introducción y Descripción General del Proceso ETL

En este notebook implementamos en **PySpark** el proceso completo de **Extracción, Transformación y Carga (ETL)** para poblar el modelo multidimensional propuesto por **RaSA**, a partir de las fuentes revisadas en la base de datos transaccional `RaSaTransaccional_ETL`:
- **F1:** `AreasDeServicio`
- **F2:** `Beneficios`
- **F3:** `BeneficiosPlanes`
- **F4:** `NivelDeRed` y `CondicionesDePago`

### Arquitectura del Modelo Multidimensional Objetivo y Orden de Carga
Dado que las tablas de hechos dependen de las llaves subrogadas (`_DWH`) de las dimensiones, el proceso ETL ejecuta los siguientes **11 bloques en orden de dependencia referencial**:

1. **Bloque 1 — Dimensión `Fecha`:** Calendario unificado con `IdFecha` (`YYYYMMDD`), `Fecha`, `Annio`, `Mes`, `Dia`. Completa las fechas que solo reportan año usando `01-01` (inicio de vigencia / creación) y `12-31` (fin de vigencia), además de incluir la `FechaMax` (`2199-12-31`) y el registro centinela (`IdFecha = 0`).
2. **Bloque 2 — Dimensión `Proveedor`:** Dimensión Tipo 1 simplificada con `IdProveedor_DWH` e `IdProveedor_T` (Nota 2 del modelo), más el registro desconocido (`0`).
3. **Bloque 3 — Dimensión `NivelesDeServicio`:** Extraída de `NivelDeRed` (F4) con `IdNivelDeServicio_DWH`, `IdNivelDeServicio_T` y `Descripcion`.
4. **Bloque 4 — Dimensión `CondicionDePago`:** Dimensión *Role-Playing* (Copago y Coseguro) a partir de `CondicionesDePago` (F4). Corrige los errores de consistencia del atributo `Tipo` para dejar únicamente `"Copago"` y `"Coseguro"`.
5. **Bloque 5 — Dimensión `TiposBeneficio`:** Dimensión maestra de beneficios (`IdTipoBeneficio_DWH`, `IdTipoBeneficio_T`, `Nombre`) bajo el esquema **SCD Tipo 4**.
6. **Bloque 6 — Minidimensión `MiniCondicionesTipoBeneficio`:** Almacena las combinaciones únicas de las 5 condiciones booleanas (estandarizadas a `"Yes"` y `"No"`) y `UnidadDelLimite` (`IdCondicionesBeneficios_DWH`).
7. **Bloque 7 — Dimensión `AreasDeServicio`:** Contiene `IdAreaDeServicio_DWH`, `IdAreaDeServicio_T`, `Nombre` y `AnnioCreacion`. Incluye obligatoriamente el registro desconocido (`IdAreaDeServicio_DWH = 0`) para no perder los planes de F3 cuyo `IdAreaDeServicio` no está en F1.
8. **Bloque 8 — Dimensión `Geografia`:** Contiene la información de `Estado`, `Condado`, `AreaAct`, `DensidadAct` y `PoblacionAct`. Aplica las dos reglas de limpieza de RaSA: multiplicar por `-1` las áreas negativas y eliminar los últimos 4 dígitos de las poblaciones terminadas en `0001` (ej. `27540001` $
ightarrow$ `2754`).
9. **Bloque 9 — Tabla Puente `AsociacionAreaServicioGeografia`:** Resuelve la relación muchos a muchos entre `AreasDeServicio` (`IdAreaDeServicio_DWH`) y `Geografia` (`IdGeografia_DWH`).
10. **Bloque 10 — Tabla de Hechos Histórica (*Factless*) `HechoHistCondicionesTiposBeneficio`:** Registra la historia de cambios de condiciones por tipo de beneficio (`IdTipoBeneficio_DWH`, `IdCondicionesTipoBeneficio_DWH`, `IdFechaInicio`, `IdFechaFin`) con la medida constante `Cambio = 1`.
11. **Bloque 11 — Tabla de Hechos Principal `HechoPlanesTiposBeneficio`:** Registra la emisión de planes con `IdPlan` como dimensión degenerada, cruza todas las llaves subrogadas, estandariza fechas al formato `YYYY-MM-DD`, imputa `333` cuando `TieneLimiteCuantitativo = 'Yes'` y `CantidadLimite` es nulo o `0`, y valida los topes de 2018 (`ValorCopago <= 3500` y `ValorCoseguro <= 100`).

---
## 2. Configuración del Entorno PySpark y Conexión a Bases de Datos

> **Antes de ejecutar:** Complete sus credenciales `db_user`, `db_psswd` y el nombre de su base de datos personal `db_destino`. Si vuelve a ejecutar el notebook completo, recuerde limpiar previamente las tablas en su esquema de destino dado que `guardar_db()` utiliza `mode('append')`.

In [1]:
# Configuración servidor base de datos transaccional y bodega de datos (DWH)
db_user = 'DB_202615_j_argumero'
db_psswd = '202625393'

db_host = '157.253.236.120'
db_port = 8080
db_origen = 'RaSaTransaccional_ETL'
db_destino = 'DB_202615_j_argumero'      # Reemplace por su base de datos personal (ej. Estudiante_i_XXXXXX)

if not db_user or not db_psswd or '<' in db_destino:
    raise ValueError('Complete db_user, db_psswd y db_destino antes de continuar')

jdbc_opciones = 'useConfigs=maxPerformance&connectTimeout=10000&socketTimeout=600000'

source_db_connection_string = f'jdbc:mysql://{db_host}:{db_port}/{db_origen}?{jdbc_opciones}'
dest_db_connection_string = f'jdbc:mysql://{db_host}:{db_port}/{db_destino}?rewriteBatchedStatements=true&{jdbc_opciones}'

In [3]:
import os
import sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
IN_CODESPACES = os.environ.get('CODESPACES') == 'true'

if IN_COLAB or IN_CODESPACES:
    %pip install -q pyspark==4.2.0 pandas==2.3.3 numpy==2.5.3 pyarrow==25.0.1 matplotlib==3.11.2 mysql-connector-python==26.7.0

if IN_COLAB:
    !apt-get install -y -qq openjdk-21-jdk-headless > /dev/null
    os.environ['JAVA_HOME'] = '/usr/lib/jvm/java-21-openjdk-amd64'

ENVIRONMENT = 'colab' if IN_COLAB else 'codespaces' if IN_CODESPACES else 'local'
print(f'Entorno detectado: {ENVIRONMENT}')

from pyspark.sql import functions as f, SparkSession, types as t
from pyspark.sql.functions import (
    lit, udf, col, length, isnan, when, count, regexp_replace,
    trim, lower, upper, coalesce, abs as spark_abs, substring, expr
)
from pyspark.sql.window import Window
from datetime import datetime, date
from pathlib import Path
import urllib.request

CONNECTOR_VERSION = '9.7.0'
CONNECTOR_URL = ('https://repo1.maven.org/maven2/com/mysql/mysql-connector-j/'
                 f'{CONNECTOR_VERSION}/mysql-connector-j-{CONNECTOR_VERSION}.jar')

def obtener_jar_mysql():
    env_path = os.environ.get('CONNECTOR_J_PATH', '')
    if env_path and Path(env_path).is_file():
        return Path(env_path)
    jar = Path.home() / '.miid4103' / 'jars' / f'mysql-connector-j-{CONNECTOR_VERSION}.jar'
    if not jar.exists():
        jar.parent.mkdir(parents=True, exist_ok=True)
        print(f'Descargando MySQL Connector/J {CONNECTOR_VERSION}...')
        urllib.request.urlretrieve(CONNECTOR_URL, jar)
    return jar

MYSQL_JAR_URI = obtener_jar_mysql().resolve().as_uri()

ERROR: Ignored the following yanked versions: 2.4.0
ERROR: Ignored the following versions that require a different python version: 1.21.2 Requires-Python >=3.7,<3.11; 1.21.3 Requires-Python >=3.7,<3.11; 1.21.4 Requires-Python >=3.7,<3.11; 1.21.5 Requires-Python >=3.7,<3.11; 1.21.6 Requires-Python >=3.7,<3.11; 2.5.0 Requires-Python >=3.12; 2.5.0rc1 Requires-Python >=3.12; 2.5.1 Requires-Python >=3.12; 2.5.2 Requires-Python >=3.12; 2.5.3 Requires-Python >=3.12; 2.5.4 Requires-Python >=3.12; 2.5.5 Requires-Python >=3.12
ERROR: Could not find a version that satisfies the requirement numpy==2.5.3 (from versions: 1.3.0, 1.4.1, 1.5.0, 1.5.1, 1.6.0, 1.6.1, 1.6.2, 1.7.0, 1.7.1, 1.7.2, 1.8.0, 1.8.1, 1.8.2, 1.9.0, 1.9.1, 1.9.2, 1.9.3, 1.10.0.post2, 1.10.1, 1.10.2, 1.10.4, 1.11.0, 1.11.1, 1.11.2, 1.11.3, 1.12.0, 1.12.1, 1.13.0, 1.13.1, 1.13.3, 1.14.0, 1.14.1, 1.14.2, 1.14.3, 1.14.4, 1.14.5, 1.14.6, 1.15.0, 1.15.1, 1.15.2, 1.15.3, 1.15.4, 1.16.0, 1.16.1, 1.16.2, 1.16.3, 1.16.4, 1.16.5, 1.16.6, 1.17

In [ ]:
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
import pyspark
os.environ.setdefault('SPARK_HOME', os.path.dirname(pyspark.__file__))
if os.name == 'nt' and 'HADOOP_HOME' not in os.environ:
    hadoop_home = Path.home() / '.miid4103' / 'hadoop'
    (hadoop_home / 'bin').mkdir(parents=True, exist_ok=True)
    (hadoop_home / 'bin' / 'winutils.exe').touch()
    os.environ['HADOOP_HOME'] = str(hadoop_home)

spark = (SparkSession.builder
         .master('local[*]')
         .appName('MIID4103-ETL-RaSA')
         .config('spark.driver.host', '127.0.0.1')
         .config('spark.jars', MYSQL_JAR_URI)
         .config('spark.sql.execution.arrow.pyspark.enabled', 'true')
         .config('spark.sql.shuffle.partitions', '8')
         .config('spark.ui.enabled', 'false')
         .config('spark.ui.showConsoleProgress', 'false')
         .config('spark.sql.ansi.enabled', 'false')
         .getOrCreate())
spark.sparkContext.setLogLevel('ERROR')
print('Sesión Spark iniciada correctamente.')

In [ ]:
def obtener_dataframe_de_bd(db_connection_string, sql, db_user, db_psswd):
    df_bd = spark.read.format('jdbc') \
        .option('url', db_connection_string) \
        .option('dbtable', sql) \
        .option('user', db_user) \
        .option('password', db_psswd) \
        .option('driver', 'com.mysql.cj.jdbc.Driver') \
        .load()
    return df_bd.cache()

def guardar_db(db_connection_string, df, tabla, db_user, db_psswd):
    df.select('*').write.format('jdbc') \
        .mode('append') \
        .option('url', db_connection_string) \
        .option('dbtable', tabla) \
        .option('user', db_user) \
        .option('password', db_psswd) \
        .option('driver', 'com.mysql.cj.jdbc.Driver') \
        .option('batchsize', 10000) \
        .option('numPartitions', 4) \
        .save()

def guardar_db_por_lotes(db_connection_string, df, tabla, db_user, db_psswd, tam_lote=50000):
    """Función auxiliar de cargue por lotes (recomendada en el enunciado si guardar_db tarda en hechos grandes)."""
    total_filas = df.count()
    print(f'Iniciando cargue de {total_filas:,} filas en {tabla}...')
    if total_filas <= tam_lote:
        guardar_db(db_connection_string, df, tabla, db_user, db_psswd)
    else:
        collected = df.collect()
        for inf in range(0, total_filas, tam_lote):
            sup = min(inf + tam_lote, total_filas)
            aux = spark.createDataFrame(collected[inf:sup], schema=df.schema)
            guardar_db(db_connection_string, aux, tabla, db_user, db_psswd)
            print(f'  Lote cargado: filas {inf:,} a {sup:,}')
    print(f'Cargue completado en {tabla}.')

### 2.1 Exploración Rápida de Esquemas en `RaSaTransaccional_ETL`
Verificamos los nombres exactos de las tablas y sus columnas en la base de datos suministrada para esta fase:

In [ ]:
# Inspección de tablas disponibles en RaSaTransaccional_ETL
df_tablas = obtener_dataframe_de_bd(
    source_db_connection_string,
    "(SELECT TABLE_NAME FROM information_schema.TABLES WHERE TABLE_SCHEMA = 'RaSaTransaccional_ETL') AS Temp_Tablas",
    db_user, db_psswd
)
df_tablas.show(30, truncate=False)

In [ ]:
# Extracción inicial de las 5 fuentes revisadas por RaSA (F1, F2, F3 y F4)
sql_f1_areas = f"(SELECT * FROM {db_origen}.AreasDeServicio) AS T_Areas"
sql_f2_beneficios = f"(SELECT * FROM {db_origen}.Beneficios) AS T_Beneficios"
sql_f3_planes = f"(SELECT * FROM {db_origen}.BeneficiosPlanes) AS T_Planes"
sql_f4_nivel_red = f"(SELECT * FROM {db_origen}.NivelDeRed) AS T_NivelRed"
sql_f4_cond_pago = f"(SELECT * FROM {db_origen}.CondicionesDePago) AS T_CondPago"

df_f1_raw = obtener_dataframe_de_bd(source_db_connection_string, sql_f1_areas, db_user, db_psswd)
df_f2_raw = obtener_dataframe_de_bd(source_db_connection_string, sql_f2_beneficios, db_user, db_psswd)
df_f3_raw = obtener_dataframe_de_bd(source_db_connection_string, sql_f3_planes, db_user, db_psswd)
df_f4_nivel_raw = obtener_dataframe_de_bd(source_db_connection_string, sql_f4_nivel_red, db_user, db_psswd)
df_f4_pago_raw = obtener_dataframe_de_bd(source_db_connection_string, sql_f4_cond_pago, db_user, db_psswd)

print("F1 AreasDeServicio:", df_f1_raw.count(), "filas | Columnas:", df_f1_raw.columns)
print("F2 Beneficios:", df_f2_raw.count(), "filas | Columnas:", df_f2_raw.columns)
print("F3 BeneficiosPlanes:", df_f3_raw.count(), "filas | Columnas:", df_f3_raw.columns)
print("F4 NivelDeRed:", df_f4_nivel_raw.count(), "filas | Columnas:", df_f4_nivel_raw.columns)
print("F4 CondicionesDePago:", df_f4_pago_raw.count(), "filas | Columnas:", df_f4_pago_raw.columns)

---
## 3. BLOQUE 1: Dimensión `Fecha`

### Reglas de Negocio y Transformaciones:
1. En el proceso de carga, si un atributo tipo fecha solo reporta el año (sin día ni mes, como ocurre en `AreasDeServicio` y `Beneficios`), se completa utilizando `01` o `12` como mes y `01` o `31` como día de acuerdo con el caso (`YYYY-01-01` para inicio de vigencia y `YYYY-12-31` para fin de vigencia).
2. En `BeneficiosPlanes` (F3), las fechas en formato anglosajón (`MMM DD,YYYY`, ej. `'Dec 31,2017'`) se transforman al formato estándar `'YYYY-MM-DD'`.
3. Se incluye la fecha máxima `2199-12-31` (`FechaMax`) para las vigencias abiertas en `HechoHistCondicionesTiposBeneficio` y el registro centinela `IdFecha = 0` (`1900-01-01`).

In [ ]:
# --- EXTRACCIÓN Y TRANSFORMACIÓN: DIMENSIÓN FECHA ---

# Función UDF para estandarizar cualquier formato de fecha a 'YYYY-MM-DD'
def estandarizar_fecha_str(val, mes_defecto='01', dia_defecto='01'):
    if val is None:
        return None
    s = str(val).strip()
    if not s or s.lower() == 'nan' or s.lower() == 'none':
        return None
    # Si viene como flotante tipo '2017.0'
    if s.endswith('.0'):
        s = s[:-2]
    # Caso 1: Solo año 'YYYY'
    if len(s) == 4 and s.isdigit():
        return f"{s}-{mes_defecto}-{dia_defecto}"
    # Caso 2: Formato ISO 'YYYY-MM-DD...'
    if len(s) >= 10 and s[4] == '-' and s[7] == '-':
        return s[:10]
    # Caso 3: Formato 'MMM DD,YYYY' o 'MMM DD, YYYY' (ej. 'Dec 31,2017')
    try:
        s_clean = s.replace(', ', ',')
        return datetime.strptime(s_clean, '%b %d,%Y').strftime('%Y-%m-%d')
    except Exception:
        pass
    return None

udf_fecha_inicio = udf(lambda x: estandarizar_fecha_str(x, '01', '01'), t.StringType())
udf_fecha_fin = udf(lambda x: estandarizar_fecha_str(x, '12', '31'), t.StringType())
udf_fecha_emision = udf(lambda x: estandarizar_fecha_str(x, '01', '01'), t.StringType())

# Recolectamos todas las fechas de emisión de F3 y los inicios/fines de año de F1, F2 y F3
fechas_f3 = df_f3_raw.select(udf_fecha_emision(col('Fecha')).alias('Fecha_Str')).filter(col('Fecha_Str').isNotNull())
fechas_f1_ini = df_f1_raw.select(udf_fecha_inicio(col('Fecha')).alias('Fecha_Str')).filter(col('Fecha_Str').isNotNull())
fechas_f1_fin = df_f1_raw.select(udf_fecha_fin(col('Fecha')).alias('Fecha_Str')).filter(col('Fecha_Str').isNotNull())
fechas_f2_ini = df_f2_raw.select(udf_fecha_inicio(col('Fecha')).alias('Fecha_Str')).filter(col('Fecha_Str').isNotNull())
fechas_f2_fin = df_f2_raw.select(udf_fecha_fin(col('Fecha')).alias('Fecha_Str')).filter(col('Fecha_Str').isNotNull())

# Agregamos también un calendario continuo de años presentes (2016 a 2020) + FechaMax (2199-12-31)
fechas_extra = spark.createDataFrame(
    [('2017-01-01',), ('2017-12-31',), ('2018-01-01',), ('2018-12-31',),
     ('2019-01-01',), ('2019-12-31',), ('2199-12-31',)],
    ['Fecha_Str']
)

df_fechas_unicas = (
    fechas_f3
    .union(fechas_f1_ini)
    .union(fechas_f1_fin)
    .union(fechas_f2_ini)
    .union(fechas_f2_fin)
    .union(fechas_extra)
    .distinct()
    .withColumn('Fecha_Date', f.to_date(col('Fecha_Str'), 'yyyy-MM-dd'))
    .filter(col('Fecha_Date').isNotNull())
)

dim_fecha = (
    df_fechas_unicas
    .select(
        f.date_format(col('Fecha_Date'), 'yyyyMMdd').cast(t.IntegerType()).alias('IdFecha'),
        col('Fecha_Str').alias('Fecha'),
        f.year(col('Fecha_Date')).cast(t.IntegerType()).alias('Annio'),
        f.month(col('Fecha_Date')).cast(t.IntegerType()).alias('Mes'),
        f.dayofmonth(col('Fecha_Date')).cast(t.IntegerType()).alias('Dia')
    )
    .distinct()
)

# Registro centinela IdFecha = 0
fecha_0 = spark.createDataFrame(
    [(0, '1900-01-01', 1900, 1, 1)],
    ['IdFecha', 'Fecha', 'Annio', 'Mes', 'Dia']
)

dim_fecha = dim_fecha.unionByName(fecha_0).dropDuplicates(['IdFecha']).orderBy('IdFecha').cache()
print(f"Total registros en Dimensión Fecha: {dim_fecha.count()}")
dim_fecha.show(15, truncate=False)

In [ ]:
# --- CARGA: DIMENSIÓN FECHA ---
guardar_db(dest_db_connection_string, dim_fecha, f'{db_destino}.Fecha', db_user, db_psswd)

---
## 4. BLOQUE 2: Dimensión `Proveedor`

### Reglas de Negocio y Transformaciones:
- Según el enunciado y la Nota 2 del modelo: *"A nivel de la dimensión Proveedor, solo se tendrán los identificadores, a pesar de que dicha dimensión tiene otros atributos. Esto por razones de simplificación del caso."*
- Se extraen los identificadores únicos `idProveedor_T` desde `BeneficiosPlanes` (F3), se genera `IdProveedor_DWH` y se añade el registro centinela `0`.

In [ ]:
# --- EXTRACCIÓN Y TRANSFORMACIÓN: DIMENSIÓN PROVEEDOR ---
df_prov = (
    df_f3_raw
    .select(col('idProveedor_T').cast(t.IntegerType()).alias('IdProveedor_T'))
    .filter(col('IdProveedor_T').isNotNull())
    .distinct()
    .orderBy('IdProveedor_T')
)

w_prov = Window.orderBy('IdProveedor_T')
df_prov = df_prov.withColumn('IdProveedor_DWH', f.row_number().over(w_prov).cast(t.IntegerType()))

prov_0 = spark.createDataFrame([(0, 0)], ['IdProveedor_DWH', 'IdProveedor_T'])

dim_proveedor = (
    df_prov.select('IdProveedor_DWH', 'IdProveedor_T')
    .unionByName(prov_0)
    .orderBy('IdProveedor_DWH')
    .cache()
)

print(f"Total registros en Dimensión Proveedor: {dim_proveedor.count()}")
dim_proveedor.show(10, truncate=False)

In [ ]:
# --- CARGA: DIMENSIÓN PROVEEDOR ---
guardar_db(dest_db_connection_string, dim_proveedor, f'{db_destino}.Proveedor', db_user, db_psswd)

---
## 5. BLOQUE 3: Dimensión `NivelesDeServicio`

### Reglas de Negocio y Transformaciones:
- Se extrae de la tabla `NivelDeRed` (F4), eliminando duplicados exactos y normalizando los campos a `IdNivelDeServicio_DWH`, `IdNivelDeServicio_T` y `Descripcion`.
- Se agrega el registro centinela `IdNivelDeServicio_DWH = 0`.

In [ ]:
# --- EXTRACCIÓN Y TRANSFORMACIÓN: DIMENSIÓN NIVELES DE SERVICIO ---
# Identificamos dinámicamente el nombre de la columna ID y Descripción en NivelDeRed
cols_nivel = df_f4_nivel_raw.columns
col_id_nivel = [c for c in cols_nivel if 'id' in c.lower()][0]
col_desc_nivel = [c for c in cols_nivel if c != col_id_nivel][0]

df_niveles = (
    df_f4_nivel_raw
    .dropDuplicates()
    .select(
        col(col_id_nivel).cast(t.IntegerType()).alias('IdNivelDeServicio_T'),
        trim(col(col_desc_nivel).cast(t.StringType())).alias('Descripcion')
    )
    .filter(col('IdNivelDeServicio_T').isNotNull())
    .dropDuplicates(['IdNivelDeServicio_T'])
    .fillna({'Descripcion': 'Sin descripción'})
    .orderBy('IdNivelDeServicio_T')
)

w_niv = Window.orderBy('IdNivelDeServicio_T')
df_niveles = df_niveles.withColumn('IdNivelDeServicio_DWH', f.row_number().over(w_niv).cast(t.IntegerType()))

nivel_0 = spark.createDataFrame(
    [(0, 0, 'Desconocido')],
    ['IdNivelDeServicio_DWH', 'IdNivelDeServicio_T', 'Descripcion']
)

dim_niveles_servicio = (
    df_niveles.select('IdNivelDeServicio_DWH', 'IdNivelDeServicio_T', 'Descripcion')
    .unionByName(nivel_0)
    .orderBy('IdNivelDeServicio_DWH')
    .cache()
)

print(f"Total registros en Dimensión NivelesDeServicio: {dim_niveles_servicio.count()}")
dim_niveles_servicio.show(15, truncate=False)

In [ ]:
# --- CARGA: DIMENSIÓN NIVELES DE SERVICIO ---
guardar_db(dest_db_connection_string, dim_niveles_servicio, f'{db_destino}.NivelesDeServicio', db_user, db_psswd)

---
## 6. BLOQUE 4: Dimensión `CondicionDePago`

### Reglas de Negocio y Transformaciones:
- RaSA informa que en `CondicionesDePago` (F4):
  1. Se eliminaron los registros con el mismo tipo y descripción que tenían diferentes identificadores.
  2. Se actualizaron los registros con `Tipo` en `NaN`.
  3. **Siguen existiendo errores de consistencia en `Tipo`, por lo que se pide corregir el tipo a solo `"Copago"` y `"Coseguro"`** y eliminar posibles duplicados.

In [ ]:
# --- EXTRACCIÓN Y TRANSFORMACIÓN: DIMENSIÓN CONDICION DE PAGO ---
print("Valores distintos de Tipo antes de la transformación:")
df_f4_pago_raw.select('Tipo').distinct().show(truncate=False)

# Diccionario de reemplazo y reglas basadas en Descripción cuando Tipo es inconsistente (ej. 'Copagado', 'SinTipoCopago')
df_cond_pago = (
    df_f4_pago_raw
    .dropDuplicates()
    .withColumn('Descripcion', trim(col('Descripcion')))
    .withColumn('Tipo_Trim', trim(col('Tipo')))
    .withColumn(
        'Tipo_Limpio',
        when(lower(col('Tipo_Trim')).isin('copago', 'copagado', 'copay'), lit('Copago'))
        .when(lower(col('Tipo_Trim')).isin('coseguro', 'coaseguro', 'coinsurance'), lit('Coseguro'))
        .when(lower(col('Descripcion')).contains('copay') | lower(col('Descripcion')).contains('copago'), lit('Copago'))
        .when(lower(col('Descripcion')).contains('coinsurance') | lower(col('Descripcion')).contains('coseguro'), lit('Coseguro'))
        .otherwise(lit('Copago'))
    )
    .select(
        col('idCondicionesDePago_T').cast(t.IntegerType()).alias('IdCondicionDePago_T'),
        col('Descripcion'),
        col('Tipo_Limpio').alias('Tipo')
    )
    .filter(col('IdCondicionDePago_T').isNotNull())
    .dropDuplicates(['IdCondicionDePago_T'])
    .orderBy('IdCondicionDePago_T')
)

print("Valores distintos de Tipo después de la transformación:")
df_cond_pago.select('Tipo').distinct().show(truncate=False)

w_pago = Window.orderBy('IdCondicionDePago_T')
df_cond_pago = df_cond_pago.withColumn('IdCondicionDePago_DWH', f.row_number().over(w_pago).cast(t.IntegerType()))

pago_0 = spark.createDataFrame(
    [(0, 0, 'Desconocido', 'Desconocido')],
    ['IdCondicionDePago_DWH', 'IdCondicionDePago_T', 'Descripcion', 'Tipo']
)

dim_condicion_pago = (
    df_cond_pago.select('IdCondicionDePago_DWH', 'IdCondicionDePago_T', 'Descripcion', 'Tipo')
    .unionByName(pago_0)
    .orderBy('IdCondicionDePago_DWH')
    .cache()
)

print(f"Total registros en Dimensión CondicionDePago: {dim_condicion_pago.count()}")
dim_condicion_pago.show(25, truncate=False)

In [ ]:
# --- CARGA: DIMENSIÓN CONDICION DE PAGO ---
guardar_db(dest_db_connection_string, dim_condicion_pago, f'{db_destino}.CondicionDePago', db_user, db_psswd)

---
## 7. BLOQUES 5 y 6: Dimensión `TiposBeneficio` y Minidimensión `MiniCondicionesTipoBeneficio` (SCD Tipo 4)

### Reglas de Negocio y Transformaciones:
1. En la fuente `Beneficios` (F2), RaSA corrigió los registros con diferentes unidades de límite y condiciones para el mismo beneficio en el mismo año, pero **siguen existiendo errores de consistencia en los valores de las condiciones (`True`, `False`, `Si`, `Nein`), por lo que solicita transformarlos para que queden únicamente `"Yes"` y `"No"`**.
2. Para el manejo de historia **Tipo 4**:
   - **`TiposBeneficio`:** Contiene `IdTipoBeneficio_DWH`, `IdTipoBeneficio_T` y `Nombre`.
   - **`MiniCondicionesTipoBeneficio`:** Contiene `IdCondicionesBeneficios_DWH`, `EstaCubiertoPorSeguro`, `EsEHB`, `TieneLimiteCuantitativo`, `ExcluidoDelDesembolsoMaximoDentroDeLaRed`, `ExcluidoDelDesembolsoMaximoFueraDeLaRed` y `UnidadDelLimite`.

In [ ]:
# --- LIMPIEZA Y ESTANDARIZACIÓN DE LA FUENTE F2 (BENEFICIOS) ---
cols_condiciones = [
    'EstaCubiertoPorSeguro',
    'EsEHB',
    'TieneLimiteCuantitativo',
    'ExcluidoDelDesembolsoMaximoDentroDeLaRed',
    'ExcluidoDelDesembolsoMaximoFueraDeLaRed'
]

# Diccionario de reemplazo para estandarizar a 'Yes' y 'No' (como recomienda la guía del curso)
mapa_yes_no = {
    'True': 'Yes', 'true': 'Yes', 'TRUE': 'Yes', 'Si': 'Yes', 'Sí': 'Yes', 'si': 'Yes', 'YES': 'Yes', 'yes': 'Yes', '1': 'Yes',
    'False': 'No', 'false': 'No', 'FALSE': 'No', 'Nein': 'No', 'nein': 'No', 'NO': 'No', 'no': 'No', '0': 'No'
}

df_f2_limpia = df_f2_raw.dropDuplicates()

for c_cond in cols_condiciones:
    df_f2_limpia = (
        df_f2_limpia
        .withColumn(c_cond, trim(col(c_cond).cast(t.StringType())))
        .na.replace(mapa_yes_no, subset=[c_cond])
        .withColumn(c_cond, when(col(c_cond).isin('Yes', 'No'), col(c_cond)).otherwise(lit('No')))
    )

# Limpieza de UnidadDelLimite y Fecha (año)
df_f2_limpia = (
    df_f2_limpia
    .withColumn('Nombre', trim(col('Nombre')))
    .withColumn(
        'UnidadDelLimite',
        when(col('UnidadDelLimite').isNull() | (trim(col('UnidadDelLimite')) == ''), lit('No aplica'))
        .otherwise(trim(col('UnidadDelLimite')))
    )
    .withColumn('Anio_Vigencia', substring(trim(col('Fecha').cast(t.StringType())), 1, 4).cast(t.IntegerType()))
    .filter(col('idTipoBeneficio_T').isNotNull())
    # Garantizar regla de RaSA: 1 solo registro por idTipoBeneficio_T y año
    .dropDuplicates(['idTipoBeneficio_T', 'Anio_Vigencia'])
    .cache()
)

print("Verificación de dominios limpios ('Yes'/'No') en F2:")
for c_cond in cols_condiciones:
    vals = [r[0] for r in df_f2_limpia.select(c_cond).distinct().collect()]
    print(f"  {c_cond}: {vals}")

In [ ]:
# --- BLOQUE 5: CONSTRUCCIÓN Y CARGA DE DIMENSIÓN TIPOSBENEFICIO ---
df_tipos_ben = (
    df_f2_limpia
    .select(
        col('idTipoBeneficio_T').cast(t.IntegerType()).alias('IdTipoBeneficio_T'),
        col('Nombre')
    )
    .dropDuplicates(['IdTipoBeneficio_T'])
    .orderBy('IdTipoBeneficio_T')
)

w_tb = Window.orderBy('IdTipoBeneficio_T')
df_tipos_ben = df_tipos_ben.withColumn('IdTipoBeneficio_DWH', f.row_number().over(w_tb).cast(t.IntegerType()))

tb_0 = spark.createDataFrame(
    [(0, 0, 'Desconocido')],
    ['IdTipoBeneficio_DWH', 'IdTipoBeneficio_T', 'Nombre']
)

dim_tipos_beneficio = (
    df_tipos_ben.select('IdTipoBeneficio_DWH', 'IdTipoBeneficio_T', 'Nombre')
    .unionByName(tb_0)
    .orderBy('IdTipoBeneficio_DWH')
    .cache()
)

print(f"Total registros en Dimensión TiposBeneficio: {dim_tipos_beneficio.count()}")
dim_tipos_beneficio.show(10, truncate=False)

guardar_db(dest_db_connection_string, dim_tipos_beneficio, f'{db_destino}.TiposBeneficio', db_user, db_psswd)

In [ ]:
# --- BLOQUE 6: CONSTRUCCIÓN Y CARGA DE MINICONDICIONESTIPOBENEFICIO ---
cols_mini = [
    'EstaCubiertoPorSeguro',
    'EsEHB',
    'TieneLimiteCuantitativo',
    'ExcluidoDelDesembolsoMaximoDentroDeLaRed',
    'ExcluidoDelDesembolsoMaximoFueraDeLaRed',
    'UnidadDelLimite'
]

df_mini = (
    df_f2_limpia
    .select(*cols_mini)
    .distinct()
    .orderBy(*cols_mini)
)

w_mini = Window.orderBy(*cols_mini)
df_mini = df_mini.withColumn('IdCondicionesBeneficios_DWH', f.row_number().over(w_mini).cast(t.IntegerType()))

mini_0 = spark.createDataFrame(
    [(0, 'Desconocido', 'Desconocido', 'Desconocido', 'Desconocido', 'Desconocido', 'Desconocido')],
    ['IdCondicionesBeneficios_DWH'] + cols_mini
)

dim_mini_condiciones = (
    df_mini.select(['IdCondicionesBeneficios_DWH'] + cols_mini)
    .unionByName(mini_0)
    .orderBy('IdCondicionesBeneficios_DWH')
    .cache()
)

print(f"Total perfiles únicos en MiniCondicionesTipoBeneficio: {dim_mini_condiciones.count()}")
dim_mini_condiciones.show(15, truncate=False)

guardar_db(dest_db_connection_string, dim_mini_condiciones, f'{db_destino}.MiniCondicionesTipoBeneficio', db_user, db_psswd)

---
## 8. BLOQUES 7, 8 y 9: `AreasDeServicio`, `Geografia` y `AsociacionAreaServicioGeografia`

### Reglas de Negocio y Transformaciones:
En la fuente `AreasDeServicio` (F1), RaSA informa que:
1. Se eliminaron los registros con diferente `IdGeografia` cuyo contenido en los demás atributos era el mismo, y se corrigieron los nulos e inválidos de fechas (1800).
2. **Áreas negativas:** Se deben corregir los registros con `Area < 0` multiplicando por `-1` esos valores.
3. **Error de digitación `0001` en `PoblacionAct`:** Los registros que terminan con 3 ceros y un uno al final (`0001`) deben corregirse quitándoles estos últimos 4 dígitos (de tal forma que si el valor era `27540001` debe quedar `2754`).
4. **Registro desconocido (`IdAreaDeServicio_DWH = 0`):** En `BeneficiosPlanes` todavía existen `IdAreaDeServicio` que no están presentes en `AreasDeServicio`, por lo que se crea el registro desconocido (`0`) para no omitir esa información.

In [ ]:
# --- LIMPIEZA INTEGRAL DE LA FUENTE F1 (AREAS DE SERVICIO Y GEOGRAFÍA) ---
# Detectamos el nombre exacto de las columnas idArea e idGeografia en F1
col_id_area_f1 = [c for c in df_f1_raw.columns if 'area' in c.lower() and 'id' in c.lower()][0]
col_id_geo_f1 = [c for c in df_f1_raw.columns if 'geo' in c.lower() and 'id' in c.lower()][0]

df_f1_limpia = (
    df_f1_raw
    .dropDuplicates()
    .filter(col(col_id_area_f1).isNotNull() & col(col_id_geo_f1).isNotNull())
    # Regla 1: Multiplicar por -1 las áreas negativas
    .withColumn(
        'Area_Limpia',
        when(col('Area') < 0, col('Area') * -1).otherwise(col('Area')).cast(t.DoubleType())
    )
    # Regla 2: Quitar los últimos 4 dígitos a PoblacionAct cuando termina en '0001' (ej. 27540001 -> 2754)
    .withColumn('Pob_Str', col('PoblacionAct').cast(t.LongType()).cast(t.StringType()))
    .withColumn(
        'PoblacionAct_Limpia',
        when(
            (length(col('Pob_Str')) > 4) & col('Pob_Str').endswith('0001'),
            substring(col('Pob_Str'), 1, length(col('Pob_Str')) - 4).cast(t.IntegerType())
        ).otherwise(col('PoblacionAct').cast(t.IntegerType()))
    )
    .withColumn('AnnioCreacion', substring(trim(col('Fecha').cast(t.StringType())), 1, 4).cast(t.IntegerType()))
    .cache()
)

print("Verificación de limpieza en F1:")
print("  Mínimo de Area_Limpia (no debe ser negativo):", df_f1_limpia.agg({'Area_Limpia': 'min'}).collect()[0][0])
print("  Máximo de PoblacionAct_Limpia (sin sufijo 0001):", df_f1_limpia.agg({'PoblacionAct_Limpia': 'max'}).collect()[0][0])

In [ ]:
# --- BLOQUE 7: CONSTRUCCIÓN Y CARGA DE DIMENSIÓN AREASDESERVICIO ---
df_areas = (
    df_f1_limpia
    .select(
        col(col_id_area_f1).cast(t.IntegerType()).alias('IdAreaDeServicio_T'),
        trim(col('NombreAreaDeServicio')).alias('Nombre'),
        col('AnnioCreacion')
    )
    .dropDuplicates(['IdAreaDeServicio_T'])
    .orderBy('IdAreaDeServicio_T')
)

w_area = Window.orderBy('IdAreaDeServicio_T')
df_areas = df_areas.withColumn('IdAreaDeServicio_DWH', f.row_number().over(w_area).cast(t.IntegerType()))

# Registro desconocido (Identificador = 0) requerido explícitamente por RaSA
area_0 = spark.createDataFrame(
    [(0, 0, 'Desconocido', 1900)],
    ['IdAreaDeServicio_DWH', 'IdAreaDeServicio_T', 'Nombre', 'AnnioCreacion']
)

dim_areas_servicio = (
    df_areas.select('IdAreaDeServicio_DWH', 'IdAreaDeServicio_T', 'Nombre', 'AnnioCreacion')
    .unionByName(area_0)
    .orderBy('IdAreaDeServicio_DWH')
    .cache()
)

print(f"Total registros en Dimensión AreasDeServicio: {dim_areas_servicio.count()}")
dim_areas_servicio.show(10, truncate=False)

guardar_db(dest_db_connection_string, dim_areas_servicio, f'{db_destino}.AreasDeServicio', db_user, db_psswd)

In [ ]:
# --- BLOQUE 8: CONSTRUCCIÓN Y CARGA DE DIMENSIÓN GEOGRAFIA ---
df_geo = (
    df_f1_limpia
    .select(
        col(col_id_geo_f1).cast(t.IntegerType()).alias('IdGeografia_T'),
        trim(col('Estado')).alias('Estado'),
        trim(col('Condado')).alias('Condado'),
        col('Area_Limpia').alias('AreaAct'),
        col('Densidad').cast(t.DoubleType()).alias('DensidadAct'),
        col('PoblacionAct_Limpia').alias('PoblacionAct')
    )
    .dropDuplicates(['IdGeografia_T'])
    .orderBy('IdGeografia_T')
)

w_geo = Window.orderBy('IdGeografia_T')
df_geo = df_geo.withColumn('IdGeografia_DWH', f.row_number().over(w_geo).cast(t.IntegerType()))

geo_0 = spark.createDataFrame(
    [(0, 0, 'Desconocido', 'Desconocido', 0.0, 0.0, 0)],
    ['IdGeografia_DWH', 'IdGeografia_T', 'Estado', 'Condado', 'AreaAct', 'DensidadAct', 'PoblacionAct']
)

dim_geografia = (
    df_geo.select('IdGeografia_DWH', 'IdGeografia_T', 'Estado', 'Condado', 'AreaAct', 'DensidadAct', 'PoblacionAct')
    .unionByName(geo_0)
    .orderBy('IdGeografia_DWH')
    .cache()
)

print(f"Total registros en Dimensión Geografia: {dim_geografia.count()}")
dim_geografia.show(10, truncate=False)

guardar_db(dest_db_connection_string, dim_geografia, f'{db_destino}.Geografia', db_user, db_psswd)

In [ ]:
# --- BLOQUE 9: CONSTRUCCIÓN Y CARGA DE TABLA PUENTE ASOCIACIONAREASERVICIOGEOGRAFIA ---
df_pares_f1 = (
    df_f1_limpia
    .select(
        col(col_id_area_f1).cast(t.IntegerType()).alias('IdAreaDeServicio_T'),
        col(col_id_geo_f1).cast(t.IntegerType()).alias('IdGeografia_T')
    )
    .distinct()
)

puente_area_geo = (
    df_pares_f1.alias('p')
    .join(dim_areas_servicio.alias('a'), col('p.IdAreaDeServicio_T') == col('a.IdAreaDeServicio_T'), 'inner')
    .join(dim_geografia.alias('g'), col('p.IdGeografia_T') == col('g.IdGeografia_T'), 'inner')
    .select(
        col('a.IdAreaDeServicio_DWH'),
        col('g.IdGeografia_DWH')
    )
    .distinct()
)

puente_0 = spark.createDataFrame([(0, 0)], ['IdAreaDeServicio_DWH', 'IdGeografia_DWH'])

asociacion_area_geografia = (
    puente_area_geo
    .unionByName(puente_0)
    .dropDuplicates(['IdAreaDeServicio_DWH', 'IdGeografia_DWH'])
    .orderBy('IdAreaDeServicio_DWH', 'IdGeografia_DWH')
    .cache()
)

print(f"Total enlaces en AsociacionAreaServicioGeografia: {asociacion_area_geografia.count()}")
asociacion_area_geografia.show(10, truncate=False)

guardar_db(dest_db_connection_string, asociacion_area_geografia, f'{db_destino}.AsociacionAreaServicioGeografia', db_user, db_psswd)

---
## 9. BLOQUE 10: Tabla de Hechos (*Factless*) `HechoHistCondicionesTiposBeneficio`

### Reglas de Negocio y Transformaciones:
1. Registra la historia de cambios de condiciones de los tipos de beneficio bajo el esquema **SCD Tipo 4**.
2. Para cada `IdTipoBeneficio_T` y año en la fuente `Beneficios` (F2), se enlaza con `TiposBeneficio` (`IdTipoBeneficio_DWH`) y con `MiniCondicionesTipoBeneficio` (`IdCondicionesTipoBeneficio_DWH`).
3. Usando funciones de ventana (`Window.partitionBy('IdTipoBeneficio_DWH').orderBy('Anio_Vigencia')`):
   - Se detectan los cambios en `IdCondicionesTipoBeneficio_DWH` entre un año y el siguiente.
   - Se define `FechaInicio = 'YYYY-01-01'` del año en que inicia la condición y `FechaFin = 'YYYY-12-31'` del año previo al siguiente cambio (o `'2199-12-31'` para la vigencia actual).
4. Se cruza con la dimensión `Fecha` para obtener `IdFechaInicio` e `IdFechaFin`, y se añade la medida constante `Cambio = 1` (Nota 3 del modelo).

In [ ]:
# --- CONSTRUCCIÓN Y CARGA DE HECHOHISTCONDICIONESTIPOSBENEFICIO ---

# 1. Enlazar F2 limpia con TiposBeneficio y MiniCondicionesTipoBeneficio
cond_join_expr = [
    df_f2_limpia[c] == dim_mini_condiciones[c] for c in cols_mini
]

df_hist_base = (
    df_f2_limpia.alias('b')
    .join(dim_tipos_beneficio.alias('tb'), col('b.idTipoBeneficio_T').cast(t.IntegerType()) == col('tb.IdTipoBeneficio_T'), 'left')
    .join(dim_mini_condiciones.alias('mc'), cond_join_expr, 'left')
    .select(
        coalesce(col('tb.IdTipoBeneficio_DWH'), lit(0)).alias('IdTipoBeneficio_DWH'),
        coalesce(col('mc.IdCondicionesBeneficios_DWH'), lit(0)).alias('IdCondicionesTipoBeneficio_DWH'),
        col('b.Anio_Vigencia')
    )
    .dropDuplicates(['IdTipoBeneficio_DWH', 'Anio_Vigencia'])
)

# 2. Identificar cambios entre años consecutivos para consolidar vigencias
w_hist = Window.partitionBy('IdTipoBeneficio_DWH').orderBy('Anio_Vigencia')

df_hist_cambios = (
    df_hist_base
    .withColumn('Cond_Anterior', f.lag('IdCondicionesTipoBeneficio_DWH').over(w_hist))
    # Es inicio de una nueva vigencia si es el primer registro del beneficio o si cambió el perfil de condiciones
    .filter(col('Cond_Anterior').isNull() | (col('IdCondicionesTipoBeneficio_DWH') != col('Cond_Anterior')))
    .withColumn('Anio_Siguiente_Cambio', f.lead('Anio_Vigencia').over(w_hist))
    .withColumn('FechaInicio_Str', f.concat(col('Anio_Vigencia').cast(t.StringType()), lit('-01-01')))
    .withColumn(
        'FechaFin_Str',
        when(
            col('Anio_Siguiente_Cambio').isNotNull(),
            f.concat((col('Anio_Siguiente_Cambio') - 1).cast(t.StringType()), lit('-12-31'))
        ).otherwise(lit('2199-12-31'))
    )
)

# 3. Cruzar con Dimensión Fecha para obtener IdFechaInicio e IdFechaFin y añadir Cambio = 1
hecho_hist_condiciones = (
    df_hist_cambios.alias('h')
    .join(dim_fecha.alias('fi'), col('h.FechaInicio_Str') == col('fi.Fecha'), 'left')
    .join(dim_fecha.alias('ff'), col('h.FechaFin_Str') == col('ff.Fecha'), 'left')
    .select(
        col('h.IdTipoBeneficio_DWH'),
        col('h.IdCondicionesTipoBeneficio_DWH'),
        coalesce(col('fi.IdFecha'), lit(0)).alias('IdFechaInicio'),
        coalesce(col('ff.IdFecha'), lit(0)).alias('IdFechaFin'),
        lit(1).cast(t.IntegerType()).alias('Cambio')
    )
    .orderBy('IdTipoBeneficio_DWH', 'IdFechaInicio')
    .cache()
)

print(f"Total registros en HechoHistCondicionesTiposBeneficio: {hecho_hist_condiciones.count()}")
hecho_hist_condiciones.show(15, truncate=False)

guardar_db(dest_db_connection_string, hecho_hist_condiciones, f'{db_destino}.HechoHistCondicionesTiposBeneficio', db_user, db_psswd)

---
## 10. BLOQUE 11: Tabla de Hechos Principal `HechoPlanesTiposBeneficio`

### Reglas de Negocio y Transformaciones:
1. **Eliminación de duplicados:** Se eliminan los registros duplicados exactos en `BeneficiosPlanes` (F3).
2. **Estandarización de Fechas:** Se transforman todas las fechas al formato `'YYYY-MM-DD'` y se cruza con `Fecha` para obtener `IdFechaEmision`.
3. **Áreas de servicio no presentes en F1:** Se cruza mediante `LEFT JOIN` con `AreasDeServicio` y se asigna `IdAreaDeServicio_DWH = 0` (registro desconocido) sin omitir ningún registro.
4. **Cruce con condiciones vigentes del beneficio (`IdCondicionesTipoBeneficio_DWH`):** Se enlaza el beneficio con su condición vigente para la fecha de emisión (`IdFechaEmision BETWEEN IdFechaInicio AND IdFechaFin`).
5. **Imputación `333` en `CantidadLimite`:** Si el beneficio tiene `TieneLimiteCuantitativo == 'Yes'` y `cantidadLimite` está sin valor (`NULL`) o en `0`, se asigna como valor por defecto `333`.
6. **Topes de Copago y Coseguro en 2018:** El valor máximo de Copago y Coseguro para el año 2018 es `3500` y `100` respectivamente.

In [ ]:
# --- EXTRACCIÓN Y TRANSFORMACIÓN: HECHOPLANESTIPOSBENEFICIO ---

# Detectamos el nombre exacto de la columna de área en F3 (idAreaDeServicio_T o idAreaServicio_T)
col_id_area_f3 = [c for c in df_f3_raw.columns if 'area' in c.lower()][0]

# T1 & T2: Deduplicación y estandarización de Fecha a YYYY-MM-DD
total_f3_raw = df_f3_raw.count()
df_f3_sin_dup = df_f3_raw.dropDuplicates()
print(f"Filas en F3 antes de deduplicar: {total_f3_raw:,} | Después de deduplicar: {df_f3_sin_dup.count():,}")

df_f3_prep = (
    df_f3_sin_dup
    .withColumn('Fecha_Emision_Str', udf_fecha_emision(col('Fecha')))
    .withColumn('Anio_Emision', substring(col('Fecha_Emision_Str'), 1, 4).cast(t.IntegerType()))
)

# Preparamos el catálogo de condiciones vigentes con su bandera TieneLimiteCuantitativo
df_hist_con_bandera = (
    hecho_hist_condiciones.alias('hh')
    .join(
        dim_mini_condiciones.select('IdCondicionesBeneficios_DWH', 'TieneLimiteCuantitativo').alias('mc'),
        col('hh.IdCondicionesTipoBeneficio_DWH') == col('mc.IdCondicionesBeneficios_DWH'),
        'left'
    )
    .select(
        col('hh.IdTipoBeneficio_DWH'),
        col('hh.IdCondicionesTipoBeneficio_DWH'),
        col('hh.IdFechaInicio'),
        col('hh.IdFechaFin'),
        col('mc.TieneLimiteCuantitativo')
    )
)

# T3 a T7: Cruces dimensionales con LEFT JOIN y coalesce a 0 (registro desconocido)
df_hecho_joined = (
    df_f3_prep.alias('p')
    .join(dim_fecha.alias('f'), col('p.Fecha_Emision_Str') == col('f.Fecha'), 'left')
    .join(dim_proveedor.alias('prv'), col('p.idProveedor_T').cast(t.IntegerType()) == col('prv.IdProveedor_T'), 'left')
    .join(dim_areas_servicio.alias('a'), col(f'p.{col_id_area_f3}').cast(t.IntegerType()) == col('a.IdAreaDeServicio_T'), 'left')
    .join(dim_niveles_servicio.alias('ns'), col('p.idNivelServicio_T').cast(t.IntegerType()) == col('ns.IdNivelDeServicio_T'), 'left')
    .join(dim_tipos_beneficio.alias('tb'), col('p.idTipoBeneficio_T').cast(t.IntegerType()) == col('tb.IdTipoBeneficio_T'), 'left')
    .join(
        df_hist_con_bandera.alias('hc'),
        (col('tb.IdTipoBeneficio_DWH') == col('hc.IdTipoBeneficio_DWH')) &
        (col('f.IdFecha').between(col('hc.IdFechaInicio'), col('hc.IdFechaFin'))),
        'left'
    )
    .join(
        dim_condicion_pago.alias('cpcoseg'),
        col('p.idCondicionesDePagoCoseguro_T').cast(t.IntegerType()) == col('cpcoseg.IdCondicionDePago_T'),
        'left'
    )
    .join(
        dim_condicion_pago.alias('cpcopag'),
        col('p.idCondicionesDePagoCopago_T').cast(t.IntegerType()) == col('cpcopag.IdCondicionDePago_T'),
        'left'
    )
)

# T8 & T9: Regla de imputación 333 en CantidadLimite y validación de topes 2018 (Copago <= 3500, Coseguro <= 100)
hecho_planes_tipos_beneficio = (
    df_hecho_joined
    .withColumn(
        'CantidadLimite_Final',
        when(
            (col('hc.TieneLimiteCuantitativo') == 'Yes') &
            (col('p.cantidadLimite').isNull() | (col('p.cantidadLimite').cast(t.DoubleType()) == 0)),
            lit(333.0)
        ).otherwise(coalesce(col('p.cantidadLimite').cast(t.DoubleType()), lit(0.0)))
    )
    .withColumn(
        'ValorCopago_Final',
        when(
            (col('p.Anio_Emision') == 2018) & (col('p.valorCopago').cast(t.DoubleType()) > 3500),
            lit(3500.0)
        ).otherwise(coalesce(col('p.valorCopago').cast(t.DoubleType()), lit(0.0)))
    )
    .withColumn(
        'ValorCoseguro_Final',
        when(
            (col('p.Anio_Emision') == 2018) & (col('p.valorCoseguro').cast(t.DoubleType()) > 100),
            lit(100.0)
        ).otherwise(coalesce(col('p.valorCoseguro').cast(t.DoubleType()), lit(0.0)))
    )
    .select(
        coalesce(col('prv.IdProveedor_DWH'), lit(0)).cast(t.IntegerType()).alias('IdProveedor'),
        coalesce(col('f.IdFecha'), lit(0)).cast(t.IntegerType()).alias('IdFechaEmision'),
        trim(col('p.idPlan_T').cast(t.StringType())).alias('IdPlan'),
        coalesce(col('a.IdAreaDeServicio_DWH'), lit(0)).cast(t.IntegerType()).alias('IdAreaDeServicio_DWH'),
        coalesce(col('ns.IdNivelDeServicio_DWH'), lit(0)).cast(t.IntegerType()).alias('IdNivelDeServicio_DWH'),
        coalesce(col('tb.IdTipoBeneficio_DWH'), lit(0)).cast(t.IntegerType()).alias('IdTipoBeneficio_DWH'),
        coalesce(col('hc.IdCondicionesTipoBeneficio_DWH'), lit(0)).cast(t.IntegerType()).alias('IdCondicionesTipoBeneficio_DWH'),
        coalesce(col('cpcoseg.IdCondicionDePago_DWH'), lit(0)).cast(t.IntegerType()).alias('IdCondicionDePagoCoseguro_DWH'),
        coalesce(col('cpcopag.IdCondicionDePago_DWH'), lit(0)).cast(t.IntegerType()).alias('IdCondicionDePagoCopago_DWH'),
        col('ValorCoseguro_Final').alias('ValorCoseguro'),
        col('ValorCopago_Final').alias('ValorCopago'),
        col('CantidadLimite_Final').alias('CantidadLimite')
    )
    .cache()
)

print(f"Total registros en HechoPlanesTiposBeneficio: {hecho_planes_tipos_beneficio.count():,}")
print(f"Registros con IdAreaDeServicio_DWH = 0 (desconocido): {hecho_planes_tipos_beneficio.filter(col('IdAreaDeServicio_DWH') == 0).count():,}")
print(f"Registros con CantidadLimite = 333 (imputados): {hecho_planes_tipos_beneficio.filter(col('CantidadLimite') == 333).count():,}")
hecho_planes_tipos_beneficio.show(10, truncate=False)

In [ ]:
# --- CARGA: HECHOPLANESTIPOSBENEFICIO ---
guardar_db_por_lotes(dest_db_connection_string, hecho_planes_tipos_beneficio, f'{db_destino}.HechoPlanesTiposBeneficio', db_user, db_psswd)

---
## 11. Verificación y Controles de Calidad Post-Carga en la Bodega de Datos

Ejecutamos consultas de verificación sobre las **10 tablas cargadas** en la bodega de datos (`db_destino`) para confirmar conteos de filas, ausencia de nulos en llaves foráneas y aplicación de las reglas de negocio de RaSA:

In [ ]:
tablas_dwh = [
    'Fecha',
    'Proveedor',
    'NivelesDeServicio',
    'CondicionDePago',
    'TiposBeneficio',
    'MiniCondicionesTipoBeneficio',
    'AreasDeServicio',
    'Geografia',
    'AsociacionAreaServicioGeografia',
    'HechoHistCondicionesTiposBeneficio',
    'HechoPlanesTiposBeneficio'
]

print("=== RESUMEN DE TABLAS POBLADAS EN EL DATA WAREHOUSE ===")
for t_nom in tablas_dwh:
    sql_check = f"(SELECT COUNT(*) AS total FROM {db_destino}.{t_nom}) AS T_Chk"
    try:
        c_val = obtener_dataframe_de_bd(dest_db_connection_string, sql_check, db_user, db_psswd).collect()[0]['total']
        print(f"  {t_nom:<38} : {c_val:>10,} filas")
    except Exception as e:
        print(f"  {t_nom:<38} : Error al consultar ({e})")

---
## 12. Uso de herramientas de IA generativa

Siguiendo las reglas prácticas obligatorias del curso, a continuación se documenta el uso de herramientas de Inteligencia Artificial Generativa en esta actividad:

### 12.1 Declaración del uso
- **Herramienta utilizada:** Antigravity (Google DeepMind) / Claude.
- **Tipo de uso:** Contraste entre el modelo dimensional de la entrega previa y el modelo multidimensional oficial estandarizado del enunciado; estructuración de la plantilla de diseño ETL (`Diseno_ETL_RaSA.xlsx`); generación de esqueletos de código PySpark para manejo de historia SCD Tipo 4 (`MiniCondicionesTipoBeneficio` y `HechoHistCondicionesTiposBeneficio`) y normalización de cadenas y fechas.

### 12.2 Prompts utilizados
1. *"En la carpeta Semana 6 hay 3 archivos: Diseno_ETL_RaSA.xlsx, Modelo dimensional RaSA.pdf y la tarea de esta semana: Construir un proceso ETL.docx. Léelos todos y establezcamos un contexto para lo que viene. Por otro lado, me parece que la tarea de ETL acota todo para un modelo diferente, confírmame esto último."*
2. *"Ajustemos ambos entregables (Plantilla de Diseño ETL en Excel y Notebook de Implementación en PySpark) para revisar el paso a paso siguiendo el modelo oficial del curso y las reglas de limpieza de RaSA."*

### 12.3 Análisis crítico del resultado
- **¿Qué partes del contenido generado fueron correctas y útiles?**
  - La identificación del cambio de arquitectura entre nuestro modelo previo (SCD Tipo 2 con una sola tabla de hechos y condados agregados) y el modelo oficial propuesto por RaSA en la guía (**SCD Tipo 4** con `MiniCondicionesTipoBeneficio` + tabla *factless* `HechoHistCondicionesTiposBeneficio`, separación de `AreasDeServicio` y `Geografia` mediante la tabla puente `AsociacionAreaServicioGeografia`, y `CondicionDePago` como *Role-Playing Dimension*).
  - El uso de diccionarios de reemplazo (`na.replace`) en PySpark para estandarizar las cinco columnas booleanas de condiciones a `'Yes'` y `'No'`, así como la función de ventana (`Window.partitionBy('IdTipoBeneficio_DWH').orderBy('Anio_Vigencia')`) con `lag` y `lead` para construir los intervalos `IdFechaInicio` e `IdFechaFin` en la tabla *factless*.
- **¿Qué errores, imprecisiones o limitaciones se identificaron y cómo se corrigieron?**
  - En los datos de `AreasDeServicio`, el error de digitación `0001` al final de `PoblacionAct` no podía corregirse con una simple resta, sino truncando los últimos 4 caracteres únicamente cuando el número termina en `'0001'` y tiene más de 4 dígitos (transformando `27540001` en `2754` sin afectar poblaciones pequeñas).
  - En `CondicionesDePago`, algunos registros tenían variantes como `'SinTipoCopago'` donde el tipo real debía inferirse inspeccionando si la columna `Descripcion` hacía referencia a *Copay* (`Copago`) o *Coinsurance* (`Coseguro`).
- **¿Qué conceptos del curso permitieron evaluar o mejorar la respuesta generada?**
  - Manejo de historia **SCD Tipo 4** (separación de atributos estables en `TiposBeneficio` y atributos cambiantes en una minidimensión `MiniCondicionesTipoBeneficio`), **Tablas de Hechos sin Hechos (*Factless Fact Tables*)** con medida constante `Cambio = 1`, **Tablas Puente (*Bridge Tables*)** para relaciones muchos a muchos entre áreas y condados, **Dimensiones Degeneradas (`IdPlan`)**, **Role-Playing Dimensions (`CondicionDePago` y `Fecha`)** y preservación de **Integridad Referencial** mediante registros centinela (`ID = 0`).

### 12.4 Aportes propios del equipo
- Detectamos desde la lectura inicial del enunciado que el proceso ETL de esta semana se acotaba a un modelo multidimensional diferente al diseñado en la etapa anterior.
- Verificamos que cada una de las aclaraciones de negocio entregadas por RaSA (imputación de `333` cuando `TieneLimiteCuantitativo = 'Yes'`, áreas negativas multiplicadas por `-1`, sufijo `0001` en población, registro desconocido `0` para áreas de F3 ausentes en F1 y topes de 3500/100 en 2018) quedara trazada tanto en la plantilla `Diseno_ETL_RaSA.xlsx` como en su respectivo bloque de PySpark.

In [ ]:
spark.stop()